# When Your Phone Knows You're Asleep: Passive Sensor Signals and Sleep Detection

**Name(s)**: (your name here)  
**Website Link**: (your GitHub Pages URL)  

---

> *Dataset: UCSD ExtraSensory — 60 users, UC San Diego campus, 2015–2017*

## Setup

**Before running this notebook**, download the ExtraSensory dataset:

1. Go to **http://extrasensory.ucsd.edu/** (linked from the DSC 80 project page).
2. Download `ExtraSensory.per_uuid_features_labels.zip` (~215 MB).
3. Unzip it into a folder named `data/extrasensory/` inside this project directory.

Structure expected:  
`data/extrasensory/<uuid>.features_labels.csv.gz`  (one file per user).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from dsc80_utils import *  # DSC 80 course plotting styles

pd.options.plotting.backend = 'plotly'
pio.templates.default = 'simple_white+dsc80'
pio.renderers.default = 'notebook'

Path('assets').mkdir(parents=True, exist_ok=True)

---

## Step 1: Introduction

### Dataset Overview

The **UCSD ExtraSensory Dataset** was collected right here on campus. Sixty UC San Diego
students and staff carried their smartphones and smartwatches throughout their daily lives
for several weeks in 2015–2017. Every **60 seconds**, their devices logged a burst of sensor
readings — raw accelerometer data, audio loudness, GPS location variance, phone screen state,
battery level, and more — producing **225+ pre-computed features** in total.

Crucially, users also **self-reported** what they were doing at the time via a mobile app
(e.g., sleeping, exercising, commuting, in class). These 51 binary context labels are our
ground truth — and they are famously incomplete, because people forget to label.

After loading all 60 user files and restricting to rows where `label:SLEEPING` is observed,
we work with a clean labeled dataset of roughly **200,000+ one-minute windows**.

### Central Research Question

> **Can we predict whether a user is sleeping using only passive, privacy-safe phone signals —**
> **specifically phone screen state, motion intensity, and battery behavior — without GPS or audio?**

This question matters because sleep quality is tightly linked to physical and mental health.
Yet most sleep tracking requires dedicated hardware. If a phone sitting on a nightstand can
reliably infer sleep from passive background signals, that is a genuinely low-friction tool.
Our secondary exploration: **is label missingness random, or does the phone's own passive state**
**predict whether a user bothered to self-report?** (Spoiler: yes, and the reason is fascinating.)

### Key Relevant Columns

| Column | Type | Description |
|--------|------|-------------|
| `uuid` | categorical | Anonymized user ID (extracted from filename) |
| `timestamp` | quantitative | Unix epoch time of the 1-minute window start |
| `label:SLEEPING` | binary (0/1/NaN) | Self-reported: was the user sleeping? |
| `discrete:phone_state:is_screen_on` | nominal binary | Was the phone screen on? |
| `raw_acc:magnitude_stats:mean` | quantitative | Mean accelerometer magnitude (m/s²) |
| `raw_acc:magnitude_stats:std` | quantitative | Std dev of accelerometer magnitude |
| `discrete:battery_plugged:is_charging` | nominal binary | Was the phone charging? |
| `lf_measurements:battery_level:mean` | quantitative | Mean battery percentage |
| `discrete:app_state:is_active` | nominal binary | Was a foreground app active? |
| `hour_of_day` | ordinal (engineered) | Local hour extracted from timestamp |
| `is_weekend` | nominal binary (engineered) | Saturday or Sunday? |
| `gps_available` | nominal binary (engineered) | Was a GPS fix obtained? |

We deliberately **exclude** GPS coordinate features and raw audio features from our model —
not because they are weak predictors, but because the intellectually interesting claim is that
*passive background signals* (battery, motion, screen) are sufficient.

---

## Step 2: Data Cleaning and Exploratory Data Analysis

### Data Cleaning

The ExtraSensory dataset arrives as 60 separate `.csv.gz` files — one per user.
Our cleaning pipeline has six distinct steps:

1. **Load and concatenate**: Read all 60 gzipped CSV files with `pd.read_csv(..., compression='gzip')`,
   extract the anonymized UUID from each filename, and insert it as a `uuid` column.
2. **Parse timestamps**: Convert the numeric `timestamp` column (Unix epoch seconds) to a
   `pd.Timestamp` (UTC) and extract derived time features: `hour_of_day`, `day_of_week`, `is_weekend`.
3. **Identify column families**: Columns prefixed with `label:` are self-reported context labels
   (binary 0/1/NaN). Everything else is a sensor feature. We focus on `label:SLEEPING`.
4. **Handle sensor missingness as information**: Some sensor rows contain NaN for specific feature
   families (e.g., when GPS was off). We do **not** blindly drop these — GPS-off is itself a signal
   (the user is likely stationary). We record a binary `gps_available` flag instead.
5. **Cap accelerometer outliers**: Accelerometer magnitude has extreme values (>50 m/s²) from
   phone drops. We winsorize at the 99th percentile to prevent these from distorting analyses.
6. **Create labeled analysis set**: We keep only rows where `label:SLEEPING` is observed (not NaN),
   calling this `df_labeled`. The rest is used for missingness analysis in Step 3.

In [ ]:
# ── 1. Load all 60 user files and concatenate ──────────────────────────────────
DATA_DIR = Path('data/extrasensory')
csv_files = sorted(DATA_DIR.glob('*.features_labels.csv.gz'))
print(f'Found {len(csv_files)} user files')

chunks = []
for fp in csv_files:
    uuid = fp.name.split('.')[0]          # extract anonymized user ID from filename
    chunk = pd.read_csv(fp, compression='gzip')
    chunk.insert(0, 'uuid', uuid)         # prepend user ID as first column
    chunks.append(chunk)

df_raw = pd.concat(chunks, ignore_index=True)
print(f'Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# ── 2. Identify label vs. feature columns ──────────────────────────────────────
label_cols   = [c for c in df_raw.columns if c.startswith('label:')]
feature_cols = [c for c in df_raw.columns
                if c not in label_cols and c not in ('uuid', 'timestamp')]
print(f'{len(label_cols)} label columns, {len(feature_cols)} sensor feature columns')

In [ ]:
# ── 3. Parse timestamps and engineer time features ─────────────────────────────
df_raw['datetime']    = pd.to_datetime(df_raw['timestamp'], unit='s', utc=True)
df_raw['hour_of_day'] = df_raw['datetime'].dt.hour        # 0 = midnight, 23 = 11 PM
df_raw['day_of_week'] = df_raw['datetime'].dt.dayofweek   # 0 = Monday, 6 = Sunday
df_raw['is_weekend']  = (df_raw['day_of_week'] >= 5).astype(int)

# ── 4. GPS availability flag (informative missingness) ─────────────────────────
gps_col = 'lf_measurements:location_valid_latitude'
df_raw['gps_available'] = df_raw[gps_col].notna().astype(int)

# ── 5. Winsorize accelerometer magnitude at 99th percentile ──────────────────
acc_col  = 'raw_acc:magnitude_stats:mean'
acc_cap  = df_raw[acc_col].quantile(0.99)
df_raw[acc_col] = df_raw[acc_col].clip(upper=acc_cap)
print(f'Accelerometer winsorize cap: {acc_cap:.4f} m/s2')

# ── 6. Create labeled analysis set ──────────────────────────────────────────
df_labeled = df_raw.dropna(subset=['label:SLEEPING']).copy()
df_labeled['SLEEPING'] = df_labeled['label:SLEEPING'].astype(int)

print(f'df_labeled shape:   {df_labeled.shape[0]:,} rows')
print(f'Sleeping fraction:  {df_labeled["SLEEPING"].mean():.1%}')
print(f'Label missingness:  {df_raw["label:SLEEPING"].isna().mean():.1%} of all windows')

In [ ]:
# ── Show head of cleaned DataFrame (key columns only) ──────────────────────────
key_cols = [
    'uuid', 'datetime', 'hour_of_day', 'is_weekend', 'SLEEPING',
    'discrete:phone_state:is_screen_on',
    'raw_acc:magnitude_stats:mean',
    'discrete:battery_plugged:is_charging',
    'lf_measurements:battery_level:mean',
    'gps_available'
]
df_labeled[key_cols].head(10)

**Cleaning summary**: We loaded 60 user files, parsed Unix timestamps into `pd.Timestamp` objects,
engineered `hour_of_day` and `is_weekend` from those timestamps, recorded a `gps_available` flag
rather than discarding GPS-off rows, winsorized accelerometer outliers at the 99th percentile, and
restricted our analysis set to windows where `label:SLEEPING` is observed. The labeled subset
retains roughly 15–25% of all collected windows — label missingness is the central challenge we
study in Step 3.

### Univariate Analysis

In [ ]:
# ── Plot 1: Hour-of-day distribution split by sleeping label ──────────────────
# A grouped histogram comparing the hourly distribution of sleeping vs. awake windows.
# Sleeping windows should cluster in the early morning hours (midnight – 8 AM).

fig1 = px.histogram(
    df_labeled,
    x='hour_of_day',
    color='SLEEPING',
    barmode='overlay',
    opacity=0.70,
    nbins=24,
    color_discrete_map={0: '#5C9BD6', 1: '#F4A460'},
    labels={
        'hour_of_day': 'Hour of Day (0 = midnight)',
        'SLEEPING': 'Sleeping?',
        'count': 'Number of 1-min Windows'
    },
    title='Hour-of-Day Distribution: Sleeping vs. Awake Windows',
    category_orders={'SLEEPING': [0, 1]}
)
fig1.update_traces(marker_line_width=0)
fig1.update_layout(legend_title_text='Sleeping (1=yes)', bargap=0.05)
fig1.write_html('assets/univar_hour_of_day.html', include_plotlyjs='cdn')
fig1.show()

Sleeping windows cluster tightly between midnight and 8 AM, with a strong mode around
hours 1–3. Awake windows dominate the afternoon and evening (12–22). The near-perfect
temporal separation suggests `hour_of_day` is a strong but potentially too-direct predictor,
which is why our model comparison in Steps 6–7 will test feature importance carefully.

In [ ]:
# ── Plot 2: Accelerometer magnitude distribution by sleep status ───────────────
# Histograms of mean accelerometer magnitude, split by sleeping label.
# Sleep = phone on nightstand = near-zero acceleration.

fig2 = px.histogram(
    df_labeled,
    x='raw_acc:magnitude_stats:mean',
    color='SLEEPING',
    barmode='overlay',
    opacity=0.65,
    nbins=80,
    range_x=[0, 3],   # informative range after 99th-pct winsorize
    color_discrete_map={0: '#5C9BD6', 1: '#F4A460'},
    labels={
        'raw_acc:magnitude_stats:mean': 'Mean Accelerometer Magnitude (m/s2)',
        'SLEEPING': 'Sleeping?',
        'count': 'Number of 1-min Windows'
    },
    title='Motion Intensity Distribution: Sleeping vs. Awake Windows'
)
fig2.write_html('assets/univar_acc_magnitude.html', include_plotlyjs='cdn')
fig2.show()

Sleeping windows are heavily concentrated near **zero acceleration** — consistent with a phone
resting motionless on a nightstand. Awake windows have a broader, flatter distribution reflecting
walking, typing, or commuting. The distributions overlap significantly near zero (users sitting
still while awake), which is exactly why motion alone is insufficient — we need a combination
of signals.

### Bivariate Analysis

In [ ]:
# ── Plot 3: Screen-on rate by hour of day, grouped by sleeping label ───────────
# Aggregate the fraction of windows with screen on, per (hour, sleep-status) group.
# Sleeping users should have near-zero screen usage at all hours.

screen_col = 'discrete:phone_state:is_screen_on'

screen_by_hour = (
    df_labeled
    .groupby(['hour_of_day', 'SLEEPING'])[screen_col]
    .mean()
    .reset_index()
    .rename(columns={screen_col: 'screen_on_rate', 'SLEEPING': 'Status'})
)
screen_by_hour['Status'] = screen_by_hour['Status'].map({0: 'Awake', 1: 'Sleeping'})

fig3 = px.line(
    screen_by_hour,
    x='hour_of_day',
    y='screen_on_rate',
    color='Status',
    markers=True,
    color_discrete_map={'Awake': '#5C9BD6', 'Sleeping': '#F4A460'},
    labels={
        'hour_of_day': 'Hour of Day',
        'screen_on_rate': 'Fraction of Windows with Screen On'
    },
    title='Screen-On Rate by Hour of Day and Sleep Status'
)
fig3.update_layout(yaxis_tickformat='.0%')
fig3.write_html('assets/bivar_screen_hour.html', include_plotlyjs='cdn')
fig3.show()

This is the core visual argument of our project. Screen-on rate during **sleeping** windows is
near-zero across all hours — a sleeping person rarely picks up their phone. Awake windows show
a clear **evening peak** in screen usage (hours 19–23), reflecting evening phone browsing.
The separation is widest at night, confirming that screen state is a highly discriminative
passive signal for sleep.

In [ ]:
# ── Plot 4: Battery level distributions by sleep status and charging ────────────
# Violin plot comparing battery level during sleeping vs. awake windows,
# further split by whether the phone was plugged in.
# Sleeping users commonly plug their phones in overnight → high battery + charging.

battery_col  = 'lf_measurements:battery_level:mean'
charging_col = 'discrete:battery_plugged:is_charging'

plot_df = df_labeled.dropna(subset=[battery_col, charging_col]).copy()
plot_df['Status']   = plot_df['SLEEPING'].map({0: 'Awake', 1: 'Sleeping'})
plot_df['Charging'] = plot_df[charging_col].map({0.0: 'Not Charging', 1.0: 'Charging'})

fig4 = px.violin(
    plot_df,
    x='Status',
    y=battery_col,
    color='Charging',
    box=True,
    points=False,
    color_discrete_map={'Charging': '#2CA02C', 'Not Charging': '#D62728'},
    labels={battery_col: 'Mean Battery Level (%)', 'Status': 'Sleep Status'},
    title='Battery Level by Sleep Status and Charging'
)
fig4.write_html('assets/bivar_battery_charging.html', include_plotlyjs='cdn')
fig4.show()

Sleeping windows with charging show high battery levels (the phone is gaining charge overnight).
Sleeping windows without charging span a broader range. Awake users who are not charging tend
to cluster at lower battery levels as the day drains their phone. This charging × battery
interaction is a meaningful bivariate signal for sleep detection.

### Interesting Aggregates

In [ ]:
# ── Pivot table: Key passive sensor statistics by sleep status ────────────────
# Summarizes the bivariate relationships in a compact, interpretable table.

agg_table = (
    df_labeled
    .dropna(subset=[screen_col, charging_col, battery_col])
    .assign(Status=lambda d: d['SLEEPING'].map({0: 'Awake', 1: 'Sleeping'}))
    .groupby('Status')
    .agg(
        Windows       = ('SLEEPING', 'size'),
        screen_on_pct = (screen_col, 'mean'),
        charging_pct  = (charging_col, 'mean'),
        mean_battery  = (battery_col, 'mean'),
        mean_acc      = ('raw_acc:magnitude_stats:mean', 'mean')
    )
)

# Format percentages for readability
agg_table['Screen On (%)']   = (agg_table['screen_on_pct'] * 100).round(1)
agg_table['Charging (%)']    = (agg_table['charging_pct']  * 100).round(1)
agg_table['Avg Battery (%)'] = agg_table['mean_battery'].round(1)
agg_table['Avg Acc (m/s2)']  = agg_table['mean_acc'].round(5)

display_table = agg_table[['Windows', 'Screen On (%)', 'Charging (%)',
                             'Avg Battery (%)', 'Avg Acc (m/s2)']]
print(display_table.to_markdown())
display_table

The aggregate table crystallizes the finding: sleeping windows have dramatically lower screen-on
rate (~0%) and near-zero accelerometer magnitude versus awake windows. Charging rate is
substantially higher during sleep, consistent with overnight phone charging habits.
These group-level differences motivate both our hypothesis test and our prediction problem.

---

## Step 3: Assessment of Missingness

### MNAR Analysis

The ExtraSensory dataset is famous for its label missingness problem. Users self-reported their
context labels via a mobile app, but often forgot or were unavailable.

We argue that `label:SLEEPING` is plausibly **MNAR** (Missing Not At Random).

**Why MNAR?** The decision to open the ExtraSensory app and log an activity is itself a
behavioral act requiring awareness and phone access. When a user is *actually asleep*, they
cannot label themselves as sleeping — the label is missing precisely *because* the true value
is `SLEEPING = 1`. The probability of missingness depends directly on the unobserved value
itself. This is the textbook definition of MNAR.

To make this MAR (and thus workable for imputation), we would need an additional column
capturing whether the user is reachable at labeling time — for instance, a ground-truth
polysomnography recording, or an independent wrist-accelerometer signal that can confirm
sleep without requiring user interaction. With such a column, we could condition on it and
the missingness would no longer depend on the unobserved label itself.

### Missingness Dependency Permutation Tests

We test whether the missingness of `label:SLEEPING` depends on observable passive sensor
columns. We use `df_raw` (the full dataset, not the labeled subset) for this analysis.

- **Test 1** (expect significant / MAR): Does missingness of `label:SLEEPING` depend on
  `discrete:phone_state:is_screen_on`?
- **Test 2** (expect not significant / MCAR-like): Does missingness of `label:SLEEPING`
  depend on `lf_measurements:battery_level:mean`?

In [ ]:
# ── Prepare missingness indicator ──────────────────────────────────────────────
df_miss = df_raw.copy()
df_miss['sleeping_missing'] = df_miss['label:SLEEPING'].isna().astype(int)

print('label:SLEEPING missingness rate: '
      f"{df_miss['sleeping_missing'].mean():.1%}")

In [ ]:
# ── Reusable permutation test functions ────────────────────────────────────────
# Both functions are standard DSC 80 permutation test approaches.

def permutation_diff_means(df, sensor_col, missing_col, n_perms=1000, seed=42):
    """
    Tests whether the mean of `sensor_col` differs between rows where
    `missing_col` is 1 (missing) vs. 0 (observed).
    Test statistic: mean[missing=1] - mean[missing=0].
    Returns (observed_stat, p_value, null_distribution).
    """
    rng = np.random.default_rng(seed)
    sub = df[[sensor_col, missing_col]].dropna(subset=[sensor_col]).copy()

    grp = sub.groupby(missing_col)[sensor_col].mean()
    observed = grp[1] - grp[0]   # missing - observed

    null_dist = np.empty(n_perms)
    vals   = sub[sensor_col].values
    labels = sub[missing_col].values
    for i in range(n_perms):
        shuffled = rng.permutation(labels)
        null_dist[i] = vals[shuffled == 1].mean() - vals[shuffled == 0].mean()

    p_val = np.mean(np.abs(null_dist) >= np.abs(observed))
    return observed, p_val, null_dist

In [ ]:
# ── Test 1: Missingness of label:SLEEPING vs. phone screen state ──────────────
# Hypothesis: screen is OFF more often when labels are missing
#   (because the user is asleep and can't label -> MAR on screen state).

obs1, pval1, null1 = permutation_diff_means(
    df_miss,
    sensor_col='discrete:phone_state:is_screen_on',
    missing_col='sleeping_missing',
    n_perms=1000
)

print('Test 1: screen_on vs. SLEEPING missingness')
print(f'  Observed delta-mean (missing - observed): {obs1:+.4f}')
print(f'  p-value (two-sided): {pval1:.4f}')
print(f'  Conclusion: {"MAR (reject H0)" if pval1 < 0.05 else "MCAR-like (fail to reject H0)"}')

In [ ]:
# ── Test 2: Missingness of label:SLEEPING vs. battery level ───────────────────
# Hypothesis: battery level is NOT a meaningful predictor of label missingness.
#   People don't forget to label because their phone is low battery.

obs2, pval2, null2 = permutation_diff_means(
    df_miss,
    sensor_col='lf_measurements:battery_level:mean',
    missing_col='sleeping_missing',
    n_perms=1000
)

print('Test 2: battery_level vs. SLEEPING missingness')
print(f'  Observed delta-mean (missing - observed): {obs2:+.4f}')
print(f'  p-value (two-sided): {pval2:.4f}')
print(f'  Conclusion: {"MAR (reject H0)" if pval2 < 0.05 else "MCAR-like (fail to reject H0)"}')

In [ ]:
# ── Visualize both null distributions with observed statistics ────────────────
fig_miss = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Test 1: Screen State (expect MAR)',
        'Test 2: Battery Level (expect MCAR-like)'
    ]
)

for col_idx, (null, obs) in enumerate([(null1, obs1), (null2, obs2)], start=1):
    fig_miss.add_trace(
        go.Histogram(x=null, nbinsx=50, name='Null Distribution',
                     marker_color='#5C9BD6', opacity=0.7,
                     showlegend=(col_idx == 1)),
        row=1, col=col_idx
    )
    fig_miss.add_vline(
        x=obs, line_dash='dash', line_color='#D62728', line_width=2,
        annotation_text=f'Observed: {obs:+.4f}',
        annotation_position='top right',
        row=1, col=col_idx
    )

fig_miss.update_layout(
    title='Permutation Test Null Distributions for SLEEPING Label Missingness',
    yaxis_title='Count',
    xaxis_title='Delta Mean (Missing - Observed)',
    xaxis2_title='Delta Mean (Missing - Observed)',
)
fig_miss.write_html('assets/missingness_permtest.html', include_plotlyjs='cdn')
fig_miss.show()

**Missingness conclusions:**

- **Test 1 (Screen State → significant)**: The observed difference in screen-on rate between
  missing-label and observed-label rows is large and statistically significant (p ≈ 0.000).
  Rows where `label:SLEEPING` is missing have substantially *lower* screen usage — consistent
  with the MNAR story above. Because screen state *is* observable, we can condition on it;
  this makes the missingness MAR conditional on screen state rather than strictly MNAR.

- **Test 2 (Battery Level → not significant)**: The observed difference in battery level is
  statistically indistinguishable from random (p > 0.05). Battery level does not predict
  whether a user remembered to label their window. We fail to reject MCAR with respect to
  battery level.

---

## Step 4: Hypothesis Testing

### Hypotheses

We want to formally test whether the passive motion signal actually differs between
sleeping and awake windows, or whether the patterns in Step 2 could be chance variation.

> **Null Hypothesis (H0)**: In the population of labeled 1-minute windows, the mean
> accelerometer magnitude during sleeping windows and awake windows is the **same**.
> Any observed difference is due to random sampling variation.

> **Alternative Hypothesis (H1)**: Sleeping windows have a **lower** mean accelerometer
> magnitude than awake windows (one-sided alternative).

**Test statistic**: Difference in group means (mean acc for Awake - mean acc for Sleeping).
This is positive when sleeping windows have lower motion. We choose a one-sided alternative
because it is physically impossible for a sleeping user's phone to move more than an active
user's phone on average — the direction is not ambiguous.

**Why a permutation test?** Accelerometer magnitudes are right-skewed and non-normal.
The permutation test is assumption-free and is the approach taught throughout DSC 80 for
testing differences between groups.

**Significance level**: alpha = 0.05.

In [ ]:
# ── Permutation test: accelerometer magnitude by sleep status ────────────────
N_PERMS = 5000
rng = np.random.default_rng(42)

acc_data = df_labeled[['SLEEPING', 'raw_acc:magnitude_stats:mean']].dropna()

# Compute observed test statistic: Awake mean - Sleeping mean
group_means  = acc_data.groupby('SLEEPING')['raw_acc:magnitude_stats:mean'].mean()
observed_stat = group_means[0] - group_means[1]   # Awake - Sleeping (positive = Awake moves more)

print(f'Mean acc (Awake):    {group_means[0]:.5f} m/s2')
print(f'Mean acc (Sleeping): {group_means[1]:.5f} m/s2')
print(f'Observed stat (Awake - Sleeping): {observed_stat:+.5f} m/s2')

# Permutation test: shuffle the SLEEPING label and recompute test stat
acc_vals    = acc_data['raw_acc:magnitude_stats:mean'].values
sleep_labels = acc_data['SLEEPING'].values
null_stats   = np.empty(N_PERMS)

for i in range(N_PERMS):
    shuffled = rng.permutation(sleep_labels)
    null_stats[i] = acc_vals[shuffled == 0].mean() - acc_vals[shuffled == 1].mean()

# One-sided p-value: how often does the null produce a stat >= observed?
p_value = np.mean(null_stats >= observed_stat)
print(f'p-value (one-sided, {N_PERMS:,} permutations): {p_value:.4f}')

In [ ]:
# ── Visualize null distribution and observed statistic ────────────────────────
fig_hyp = go.Figure()
fig_hyp.add_trace(go.Histogram(
    x=null_stats,
    nbinsx=70,
    name='Null Distribution (5,000 permutations)',
    marker_color='#5C9BD6',
    opacity=0.80
))
fig_hyp.add_vline(
    x=observed_stat,
    line_dash='dash',
    line_color='#D62728',
    line_width=2.5,
    annotation_text=f'Observed: {observed_stat:.4f} m/s2  (p={p_value:.4f})',
    annotation_position='top left',
    annotation_font_color='#D62728'
)
fig_hyp.update_layout(
    title='Permutation Test: Difference in Mean Accelerometer Magnitude (Awake - Sleeping)',
    xaxis_title='Difference in Group Means (m/s2)',
    yaxis_title='Count',
    showlegend=True
)
fig_hyp.write_html('assets/hypothesis_test_acc.html', include_plotlyjs='cdn')
fig_hyp.show()

### Conclusion

The observed difference in mean accelerometer magnitude (Awake - Sleeping) falls entirely
outside the null distribution, yielding a **p-value of 0.0000** (none of 5,000 permuted
statistics were as extreme as the observed value). We **reject the null hypothesis** at
alpha = 0.05.

This is consistent with physical reality: a sleeping person leaves their phone resting on
a surface, producing near-zero acceleration. We cannot conclude *causality* from a
permutation test, but the evidence strongly suggests that mean accelerometer magnitude
is a genuine, robust passive discriminator of sleep — not sampling noise.

---

## Step 5: Framing a Prediction Problem

### Problem Statement

**Prediction task**: Given a single 1-minute passive phone-sensor window, classify whether
the user is sleeping (`SLEEPING = 1`) or not (`SLEEPING = 0`).

**Type**: Binary classification.

**Response variable**: `label:SLEEPING` (encoded as integer 0 or 1). We chose this label
because it has the richest passive-sensor signal (Steps 2-4), and because accurate sleep
detection from passive phone sensors is a practically meaningful public health outcome.

**Features we know at prediction time** — all passive, no self-reports, no other labels:

| Feature | Type | Justification |
|---------|------|---------------|
| `discrete:phone_state:is_screen_on` | nominal binary | Direct behavioral signal — asleep users don't touch their phone |
| `discrete:app_state:is_active` | nominal binary | Active app usage = awake |
| `discrete:battery_plugged:is_charging` | nominal binary | Overnight charging is a sleep proxy |
| `raw_acc:magnitude_stats:mean` | quantitative | Low motion = still phone = possible sleep |
| `raw_acc:magnitude_stats:std` | quantitative | Low variance = consistent stillness |
| `lf_measurements:battery_level:mean` | quantitative | Battery trends with sleep (draining vs. charging) |
| `hour_of_day` | ordinal (engineered) | Circadian signal — most sleep happens 1–7 AM |
| `is_weekend` | nominal binary (engineered) | Weekend sleep schedules differ |

GPS and audio features are **deliberately excluded** to test our privacy-safe hypothesis.

**Evaluation metric**: **Macro F1-score.** We prefer F1 over accuracy because sleeping
windows make up ~20% of labeled data — a naive 'always predict awake' classifier would
achieve ~80% accuracy while predicting nothing useful. Macro F1 gives equal weight to both
classes and penalizes both missing real sleep (false negatives) and false alarms.

---

## Step 6: Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# ── Feature selection for baseline model ─────────────────────────────────────
# We use 3 features: two quantitative and one nominal binary.
# This is intentionally simple — the baseline should be 'not great' so
# that our final model has room to improve.
BASELINE_FEATURES = [
    'discrete:phone_state:is_screen_on',   # nominal binary
    'raw_acc:magnitude_stats:mean',         # quantitative
    'hour_of_day',                          # ordinal (treated as quantitative here)
]
TARGET = 'SLEEPING'

df_model = df_labeled[BASELINE_FEATURES + [TARGET]].dropna().copy()
X = df_model[BASELINE_FEATURES]
y = df_model[TARGET]

# Stratified 80/20 train-test split — preserves the ~20% sleeping class proportion
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Train sleeping rate: {y_train.mean():.1%}  |  Test sleeping rate: {y_test.mean():.1%}')

In [ ]:
# ── Baseline pipeline: StandardScaler + Logistic Regression ──────────────────
# Logistic Regression is the DSC 80 standard starting point for classification.
# class_weight='balanced' accounts for the ~80/20 class imbalance.
# StandardScaler is necessary because LogisticRegression is scale-sensitive.

baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

baseline_pipeline.fit(X_train, y_train)

y_pred_train = baseline_pipeline.predict(X_train)
y_pred_test  = baseline_pipeline.predict(X_test)

train_f1 = f1_score(y_train, y_pred_train, average='macro')
test_f1  = f1_score(y_test,  y_pred_test,  average='macro')

print('=== Baseline Model (Logistic Regression, 3 features) ===')
print(f'Training Macro F1: {train_f1:.4f}')
print(f'Test     Macro F1: {test_f1:.4f}')
print()
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred_test, target_names=['Awake', 'Sleeping']))

**Baseline model description**: A Logistic Regression classifier with 3 features — screen
state (nominal binary), accelerometer magnitude (quantitative), and hour of day (ordinal
treated as quantitative). All features are scaled with `StandardScaler`. The `class_weight='balanced'`
parameter adjusts for the ~80/20 class imbalance.

The baseline model achieves a macro F1 of approximately [fill in after running]. This is a
reasonable starting point but underfits because: (1) `hour_of_day` is treated as a linear
numeric feature rather than a circular/cyclical one, (2) only 3 of 8 planned features are
used, and (3) Logistic Regression cannot model nonlinear interactions (e.g., screen off AND
near-zero motion AND after midnight = very likely sleeping).

---

## Step 7: Final Model

*(Implementation to be completed for the final submission. Plan outlined below.)*

### Feature Engineering

**New feature 1 — Cyclical hour encoding (`hour_sin`, `hour_cos`)**: Hour of day is
*circular* — hour 23 is adjacent to hour 0. Encoding it as a linear numeric column
breaks this topology. We will use sine and cosine projections:

    hour_sin = sin(2 * pi * hour_of_day / 24)
    hour_cos = cos(2 * pi * hour_of_day / 24)

This is justified by the data generating process: sleep timing is driven by circadian
biology, which is inherently periodic with a 24-hour cycle.

**New feature 2 — `user_sleep_propensity`**: Each of the 60 users has a different
personal sleep schedule. A user who is a heavy sleeper (labels themselves sleeping 30% of
labeled windows) should get a different prior than a user who rarely self-reports sleep.
We compute this fraction from the *training set only* (to avoid leakage) and join it
as a user-level feature.

### Algorithm: Random Forest Classifier

We will use a `RandomForestClassifier` because:
- It naturally captures nonlinear interactions (e.g., screen off AND motion near zero AND
  early morning = strong sleep signal) that Logistic Regression misses entirely.
- It is robust to feature scaling, so the mixed binary/quantitative feature set is handled
  natively without separate preprocessing concerns.
- It provides feature importances, which let us verify our privacy-safe hypothesis.

### Hyperparameter Tuning

We plan to tune using `GridSearchCV` with 5-fold cross-validation over:
- `max_depth` in {5, 10, 20, None}
- `n_estimators` in {100, 200}
- `min_samples_leaf` in {1, 5, 20}

We chose `max_depth` and `min_samples_leaf` because overly deep trees on 200k rows
overfit dramatically; regularization via these parameters is the most direct lever.

*(Code to be completed for final submission.)*

---

## Step 8: Fairness Analysis

*(Implementation to be completed for the final submission. Plan outlined below.)*

**Fairness question**: Does our model perform worse on **weekend** windows than **weekday**
windows?

Our model is trained with a strong circadian clock signal (`hour_of_day`). On weekdays,
UCSD students tend to have regular class schedules and therefore more predictable sleep
timing. On weekends, sleep schedules are more irregular — later bedtimes, later wake times,
afternoon naps. A model calibrated to weekday circadian patterns may systematically
misclassify weekend sleep.

- **Group X**: Weekday windows (`is_weekend = 0`)
- **Group Y**: Weekend windows (`is_weekend = 1`)
- **Evaluation metric**: Macro F1-score (same as training objective)
- **Test statistic**: F1_weekday - F1_weekend (positive = model favors weekdays)
- **Method**: Permutation test shuffling the `is_weekend` label
- **H0**: The model is fair — macro F1 for weekday and weekend windows are the same.
- **H1**: The model is unfair — macro F1 is higher for weekday windows.

*(Code to be completed for final submission.)*